### Beyond the Squeaky Wheel: 311 Engagement & Equity Analysis
### Notebook 6: Service Need Index (SNI) Index Construction

Converts the prepped SNI metrics into the final Service Need Index. Applies percentile-rank standardization pooled across cities, weights and combines metrics into a composite score, and exports tract-level and city-level SNI summaries.

In [ ]:
# Step 1: Import libraries

import os
import pandas as pd
import numpy as np
import geopandas as gpd
from tqdm import tqdm

In [ ]:
# Step 2: Set up file paths and inputs/outputs

MASTER_INPUT_DIR = "INSERT FOLDER PATH: SNI data prep outputs"
TRACTS_FILENAME = "SNI_master_tract_variables.gpkg"

MASTER_INPUT_PATH = os.path.join(MASTER_INPUT_DIR, TRACTS_FILENAME)

OUTPUT_DIR = "INSERT FOLDER PATH: SNI index outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRACT_SCORES_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "SNI_tract_scores.csv")
CITY_SUMMARY_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "SNI_city_summary.csv")
CORRELATION_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "SNI_variable_correlations.csv")

PDF_OUTPUT_PATH = "INSERT FILE PATH: output histogram PDF"

In [ ]:
# Sanity Check: confirm input file found

assert os.path.exists(MASTER_INPUT_PATH), f"Master input not found: {MASTER_INPUT_PATH}"

if MASTER_INPUT_PATH.endswith((".gpkg", ".shp")):
    df = gpd.read_file(MASTER_INPUT_PATH, engine="pyogrio")
else:
    df = pd.read_csv(MASTER_INPUT_PATH)

print(f"Loaded {len(df):,} tracts")
print(f"Cities found: {df['city'].nunique()}")
print(df["city"].value_counts())

In [ ]:
# Step 3: Configure weight tiers

# Weight tier multipliers, adjust these as needed/methodological

WEIGHT_TIERS = {"high": 2.0, "normal": 1.0}
        # refined weight scale
        # "low" is the absence of a variable, 0 value

# Roads variable built from two weighted sub-components 
ROAD_COMPONENTS = {
    "major_roads": {"column": "major_roads_length", "tier": "high"},
    "minor_roads": {"column": "minor_roads_length", "tier": "normal"},     
}

# Zoning variable built from a weighted sum of category percentages
ZONING_COMPONENTS = {
    "industrial": {"column": "pct_industrial", "tier": "high"},
    "commercial": {"column": "pct_commercial", "tier": "normal"},
    "mixed_use": {"column": "pct_mixed_use", "tier": "normal"},
}

# Transit variable built from two unweighted sub-compenents
TRANSIT_COMPONENTS = ["railway_length", "transit_lines_length"]

# All othervariables are unweighted single percentile-ranked scores 
OTHER_VARS = {
    "impermeability": {"column": "Impervious_Prcnt", "direction": 1},
    "lodes_jobs": {"column": "total_jobs", "direction": 1},
    "housing_age": {"column": "building_age_years", "direction": 1},
    "overture_poi": {"column": "poi_count", "direction": 1},
}

# Confirm all expected raw columns are present before proceeding
expected_cols = (
    [v["column"] for v in ROAD_COMPONENTS.values()]
    + [v["column"] for v in ZONING_COMPONENTS.values()]
    + TRANSIT_COMPONENTS
    + [v["column"] for v in OTHER_VARS.values()]
)
missing_cols = [c for c in expected_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing expected columns in master table: {missing_cols}")

In [ ]:
# Step 4: Percentile-rank standardization (cross-city pooled)

# Each variable is ranked against all tracts in all cities 

def percentile_rank_0_100(series: pd.Series, direction: int = 1) -> pd.Series:
    """Percentile rank a series to a 0-100 scale. NaNs stay NaN (na_option='keep')."""
    ranked = series.rank(pct=True, method="average", na_option="keep") * 100
    if direction == -1:
        ranked = 100 - ranked
    return ranked

def flag_missing(df, col, label):
    df[f"flag_{label}_missing"] = df[col].isna()

# Roads sub-components: rank major and minor independently, then combine
for name, meta in ROAD_COMPONENTS.items():
    col = meta["column"]
    flag_missing(df, col, name)
    df[f"{name}_pctrank"] = percentile_rank_0_100(df[col])

major_w = WEIGHT_TIERS[ROAD_COMPONENTS["major_roads"]["tier"]]
minor_w = WEIGHT_TIERS[ROAD_COMPONENTS["minor_roads"]["tier"]]
df["roads_score"] = (
    df["major_roads_pctrank"] * major_w + df["minor_roads_pctrank"] * minor_w
) / (major_w + minor_w)

# Zoning sub-components: weighted sum of raw % area, then rank composite
for name, meta in ZONING_COMPONENTS.items():
    col = meta["column"]
    flag_missing(df, col, name)

zoning_weighted_sum = sum(
    df[meta["column"]] * WEIGHT_TIERS[meta["tier"]] for meta in ZONING_COMPONENTS.values()
)
df["zoning_score"] = percentile_rank_0_100(zoning_weighted_sum)

# Transit sub-components: direct sum of at-grade rail + local transit length, then rank --
for col in TRANSIT_COMPONENTS:
    flag_missing(df, col, col)

df["transit_length_total"] = df[TRANSIT_COMPONENTS].sum(axis=1, skipna=True)
df["transit_score"] = percentile_rank_0_100(df["transit_length_total"])

# Remaining variables: single percentile rank each
for var_name, meta in tqdm(OTHER_VARS.items(), desc="Percentile-ranking remaining SNI variables"):
    col = meta["column"]
    flag_missing(df, col, var_name)
    df[f"{var_name}_score"] = percentile_rank_0_100(df[col], direction=meta["direction"])

In [ ]:
# Step 5: Calculate composite SNI

# Equal-weighted average across the 7 final variables 
# re-percentile-rankedforclean, fully-spread 0-100 final score

FINAL_VARS = ["roads_score", "zoning_score", "transit_score"] + [f"{v}_score" for v in OTHER_VARS]

df["SNI_raw"] = df[FINAL_VARS].mean(axis=1, skipna=True)
df["SNI_Score"] = percentile_rank_0_100(df["SNI_raw"], direction=1)

In [ ]:
# Step 6: Redundancy / sensitivity check (OECD Handbook recommendation)

# Flag any pair of final variables with high correlation
# near-duplicate would indicators infla weight of whatever they represent.

corr_matrix = df[FINAL_VARS].corr()
corr_matrix.to_csv(CORRELATION_OUTPUT_PATH)

high_corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
high_corr_pairs.columns = ["variable_1", "variable_2", "correlation"]
high_corr_pairs = high_corr_pairs[high_corr_pairs["correlation"].abs() > 0.7]

if len(high_corr_pairs) > 0:
    print("Flagged variable pairs with |correlation| > 0.7 -- review for redundancy:")
    print(high_corr_pairs.sort_values("correlation", ascending=False))
else:
    print("No variable pairs exceed the 0.7 correlation threshold.")

In [ ]:
# Step 7: Export tract-level scores

flag_cols = [c for c in df.columns if c.startswith("flag_") and c.endswith("_missing")]
export_cols = ["STATEFP", "COUNTYFP", "GEOID", "city", "ALAND", "AWATER", "point_count"] + FINAL_VARS + flag_cols + ["SNI_raw", "SNI_Score"]
df[export_cols].to_csv(TRACT_SCORES_OUTPUT_PATH, index=False)
print(f"Tract-level SNI scores exported to {TRACT_SCORES_OUTPUT_PATH}")

In [ ]:
# Step 7: Export tract-level scores

gdf_export = tracts_gdf[["GEOID", "geometry"]].merge(df[export_cols], on="GEOID", how="inner")
gdf_export.to_file(TRACT_SCORES_OUTPUT_PATH.replace(".csv", ".gpkg"), driver="GPKG", engine="pyogrio")
print(f"Tract-level SNI scores exported to {TRACT_SCORES_OUTPUT_PATH.replace('.csv', '.gpkg')}")

In [ ]:
# Step 8: Generate city-level summary table

summary_rows = []

for city, city_df in tqdm(df.groupby("city"), desc="Summarizing by city"):
    summary_rows.append({
        "city": city,
        "n_tracts": len(city_df),
        "SNI_mean": city_df["SNI_Score"].mean(),
        "SNI_median": city_df["SNI_Score"].median(),
        "SNI_std": city_df["SNI_Score"].std(),
        "SNI_min": city_df["SNI_Score"].min(),
        "SNI_max": city_df["SNI_Score"].max(),
        **{f"{v}_mean": city_df[v].mean() for v in FINAL_VARS},
    })

summary_df = pd.DataFrame(summary_rows).sort_values("city")
summary_df.to_csv(CITY_SUMMARY_OUTPUT_PATH, index=False)

print(f"City-level SNI summary exported to {CITY_SUMMARY_OUTPUT_PATH}")
summary_df

In [ ]:
# Sanity Check: area relative to SNI score

# Is this specific to NYC's numbers, or does the pattern hold if you exclude it?
without_nyc = city_medians.drop("new_york")
print(f"Correlation excluding NYC: {without_nyc['median_ALAND'].corr(without_nyc['median_SNI']):.3f}")

# Quick visual -- does NYC look like an outlier on a scatter, not part of a trend?
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(city_medians["median_ALAND"], city_medians["median_SNI"])
for city, row in city_medians.iterrows():
    ax.annotate(city, (row["median_ALAND"], row["median_SNI"]), fontsize=8)
ax.set_xlabel("Median tract area (ALAND, m²)")
ax.set_ylabel("Median SNI Score")
plt.show()

In [ ]:
# Step 9: Generate SNI histograms by city and export PDF

# ---- HISTOGRAMS: 2x1 per city, one page per city ----
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

cities = sorted(df["city"].dropna().unique())

transform_cols = [
    ("SNI_raw", "raw"),
    ("SNI_Score", "indexed"),
]

with PdfPages(PDF_OUTPUT_PATH) as pdf:
    for city in cities:
        subset = df[df["city"] == city]

        fig, axes = plt.subplots(1, 2, figsize=(10, 6))
        axes = axes.flatten()

        for ax, (col, label) in zip(axes, transform_cols):
            ax.hist(subset[col].dropna(), bins=30, color="steelblue", edgecolor="white")
            ax.set_title(f"{city} - {label}")

        plt.suptitle(city, fontsize=14)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.show()

print(f"Saved comparison histograms to {PDF_OUTPUT_PATH}")